In [3]:

from specutils.fitting import fit_generic_continuum
from specutils.analysis import equivalent_width
from astropy.nddata import StdDevUncertainty
from specutils import SpectralRegion
from specutils import Spectrum
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from pathlib import Path
import seaborn as sns
import pandas as pd
import numpy as np
import warnings



#Define source path where the files are
source=Path("/home/nicolas/Documents/Research/PhD/JWST-Data/PRISM/")
mast_folder=source / "Spectra1D/MAST"
jades_folder=source / "Spectra1D/JADES"
output_folder=source / "EW/EW-Data-Output"
plots_folder=source / "EW/EW-Plots"
objects_folder=source / "EW/EW-Objects"

#Extract data with Pandas
df = pd.read_csv("short-table.tsv", sep='\t')
ID_array = df["NIRSpec_ID"]
JADES_files = df["JADES_FILENAME"]
MAST_files = df["MAST_FILENAME"]
z_array = df["redshift"]

#Create arrays to save the data with save_EW()
ID_data =[]
z_data = []
mast_EW_Ha_data = []
mast_EW_dHa_data =[]
mast_EW_Hb_data = []
mast_EW_dHb_data =[]
mast_EW_O_data = []
mast_EW_dO_data =[]

jades_EW_Ha_data = []
jades_EW_dHa_data =[]
jades_EW_Hb_data = []
jades_EW_dHb_data= []
jades_EW_O_data =[]
jades_EW_dO_data=[]



#Define Spectral Regions for the emission lines: this is the region
#we edit
#//////////////////////////////////////
indx=27

Ha_ini, Ha_end = 6534 *u.AA, 6606 *u.AA
Hb_ini, Hb_end = 4828 *u.AA, 4885 *u.AA
O_ini, O_end = 4912 *u.AA, 5068 *u.AA
#//////////////////////////////////////



In [4]:
index=20

Ha_ini, Ha_end = 6534 *u.AA, 6606 *u.AA
Hb_ini, Hb_end = 4828 *u.AA, 4885 *u.AA
O_ini, O_end = 4912 *u.AA, 5068 *u.AA


jades_file = jades_folder / JADES_files[index]
mast_file = mast_folder / MAST_files[index]
z = z_array[index]

#///////// Extract MAST DATA /////////
    

with fits.open(mast_file) as m_f:
    #Extract data from the file
    specdata=m_f[1].data

    #Compute rest wavelength
    lambda_obs = specdata['WAVELENGTH'] * 1e-6 / 1e-10 #convert from um to AA.
    lambda_rest = lambda_obs / (1.+float(z))
    lambda_rest_Angstrom = lambda_rest * u.AA #u.AA just includes the units

    #Extract flux in Jy from .fits
    flux = specdata['FLUX']
    
    flux_Jy = flux * u.Jy

    #Uncertainty of the flux
    flux_err = specdata['FLUX_ERROR']
    flux_err_Jy = flux_err * u.Jy
    flux_uncertainty = StdDevUncertainty(flux_err_Jy)

    #Define the spectrum over which the EW will be calculated, use class Spectrum1D from specutils
    spectrum = Spectrum(spectral_axis=lambda_rest_Angstrom,
                                flux=flux_Jy,
                                uncertainty=flux_uncertainty)


    #Remove NaN values
    mask = np.isfinite(flux_Jy)
    clean_spectrum = Spectrum(spectral_axis=lambda_rest_Angstrom[mask],
                            flux=flux_Jy[mask],
                            uncertainty=flux_uncertainty[mask])


    with warnings.catch_warnings(): #ignore warnings
        warnings.simplefilter('ignore')

        #///////// Regions to exlude /////////

        lamb = lambda_rest_Angstrom
        #For Ha
        Ha_left = SpectralRegion(lamb[0], Ha_ini - 500 *u.AA)
        Ha_region = SpectralRegion(Ha_ini, Ha_end)
        Ha_region_excl = SpectralRegion(Ha_ini, Ha_end + 150 *u.AA )
        Ha_right = SpectralRegion(Ha_end + 350 *u.AA, lamb[-1])
        Ha_exclusion_regions = [Ha_left, Ha_region_excl, Ha_right]

        #For Hb and [OIII]
        Hb_left = SpectralRegion(lamb[0], Hb_ini - 500 *u.AA)
        Hb_region = SpectralRegion(Hb_ini, Hb_end)
        OIII_region = SpectralRegion(O_ini, O_end)
        OIII_right = SpectralRegion(O_end + 500 *u.AA, lamb[-1])
        Hb_exclusion_regions = [Hb_left, Hb_region, OIII_region, OIII_right]

        #//////// Compute continuum by fitting /////////

        Ha_continuum_fit = fit_generic_continuum(clean_spectrum, exclude_regions=Ha_exclusion_regions)(clean_spectrum.spectral_axis)
        Hb_continuum_fit = fit_generic_continuum(clean_spectrum, exclude_regions=Hb_exclusion_regions)(clean_spectrum.spectral_axis)


        #////// Normalize the spectrum by its continuum ///////////

    with warnings.catch_warnings(): #ignore warnings
        warnings.simplefilter('ignore')

        #///////// Regions to exlude /////////

        lamb = lambda_rest_Angstrom
        #For Ha
        Ha_left = SpectralRegion(lamb[0], Ha_ini - 500 *u.AA)
        Ha_region = SpectralRegion(Ha_ini, Ha_end)
        Ha_region_excl = SpectralRegion(Ha_ini, Ha_end + 150 *u.AA )
        Ha_right = SpectralRegion(Ha_end + 350 *u.AA, lamb[-1])
        Ha_exclusion_regions = [Ha_left, Ha_region_excl, Ha_right]

        #For Hb and [OIII]
        Hb_left = SpectralRegion(lamb[0], Hb_ini - 500 *u.AA)
        Hb_region = SpectralRegion(Hb_ini, Hb_end)
        OIII_region = SpectralRegion(O_ini, O_end)
        OIII_right = SpectralRegion(O_end + 500 *u.AA, lamb[-1])
        Hb_exclusion_regions = [Hb_left, Hb_region, OIII_region, OIII_right]

        #//////// Compute continuum by fitting /////////

        Ha_continuum_fit = fit_generic_continuum(clean_spectrum, exclude_regions=Ha_exclusion_regions)(clean_spectrum.spectral_axis)


        #////// Normalize the spectrum by its continuum ///////////

        Ha_normalized_continuum_spec = clean_spectrum / Ha_continuum_fit

        #////// Compute Equivalent Width ///////////

        Ha_EW = equivalent_width(Ha_normalized_continuum_spec, continuum=1, regions=Ha_region)
        # Save the results

        Ha_normalized_continuum_spec = clean_spectrum / Ha_continuum_fit

        #////// Compute Equivalent Width ///////////

        Ha_EW = equivalent_width(Ha_normalized_continuum_spec, continuum=1, regions=Ha_region)
       

print(Ha_EW.value, Ha_EW.uncertainty)



Exception: Spectrum flux is empty or None.

In [8]:

import numpy as np

def std_flux(lambda_array, flux_array, regionLeft_ini, regionLeft_end, regionRight_ini, regionRight_end):
    lambda_array = np.array(lambda_array)
    flux_array = np.array(flux_array)
    
    
    # Crear máscara booleana para el rango deseado
    lambda_ini = regionLeft_ini
    lambda_end = regionLeft_end
    mask_left = (lambda_array >= lambda_ini) & (lambda_array <= lambda_end)
    
    lambda_ini = regionRight_ini
    lambda_end = regionRight_end
    mask_right = (lambda_array >= lambda_ini) & (lambda_array <= lambda_end)
    
    # Extraer los valores correspondientes
    lambda_left = lambda_array[mask_left]
    flux_left = flux_array[mask_left]
    
    #print(flux_left)
    
    lambda_right = lambda_array[mask_right]
    flux_right = flux_array[mask_right]
    
    #print(flux_right)
    stdDev_left = np.std(flux_left)
    stdDev_right = np.std(flux_right)
    
    
    
    return (stdDev_left + stdDev_right)/2

In [ ]:
#//////////////////////////////////////
index=20

Ha_ini, Ha_end = 6528 *u.AA, 6699 *u.AA
Hb_ini, Hb_end = 4810 *u.AA, 4884 *u.AA
O_ini, O_end = 4907 *u.AA, 5046 *u.AA

jades_file = jades_folder / JADES_files[index]
mast_file = mast_folder / MAST_files[index]
z = z_array[index]
#//////////////////////////////////////



with fits.open(mast_file) as m_f:
    #Extract data from the file
    specdata=m_f[1].data

    #Extract flux in Jy from .fits
    flux = specdata['FLUX']

    mask = np.isfinite(flux)
    flux = flux[mask]
    flux_Jy = flux * u.Jy


    #Compute rest wavelength
    lambda_obs = specdata['WAVELENGTH'] * 1e-6 / 1e-10 #convert from um to AA.
    lambda_rest = lambda_obs[mask] / (1.+float(z))
    lambda_rest_Angstrom = lambda_rest * u.AA #u.AA just includes the units

    

    #Uncertainty of the flux from .fits
    flux_err = specdata['FLUX_ERROR']
    flux_err_Jy = flux_err * u.Jy
    flux_uncertainty = StdDevUncertainty(flux_err_Jy)
    
    #Uncertainty of the flux from std of vector of flux
    flux_err_2 = std_flux(lambda_rest, flux, Ha_ini.value - 500, Ha_ini.value, Ha_end.value + 150, Ha_end.value + 350 )
    flux_err_Jy2 = flux_err_2 * np.ones(len(flux)) * u.Jy
    flux_uncertainty_2 = StdDevUncertainty(flux_err_Jy2)
    


    #Remove NaN values
    
    clean_spectrum = Spectrum(spectral_axis=lambda_rest_Angstrom,
                            flux=flux_Jy,
                            uncertainty=flux_uncertainty_2)
    

    #////// Normalize the spectrum by its continuum ///////////

    with warnings.catch_warnings(): #ignore warnings
        warnings.simplefilter('ignore')

        #///////// Regions to exlude /////////

        lamb = lambda_rest_Angstrom
        #For Ha
        Ha_left = SpectralRegion(lamb[0], Ha_ini - 500 *u.AA)
        Ha_region = SpectralRegion(Ha_ini, Ha_end)
        Ha_region_excl = SpectralRegion(Ha_ini, Ha_end + 150 *u.AA )
        Ha_right = SpectralRegion(Ha_end + 350 *u.AA, lamb[-1])
        Ha_exclusion_regions = [Ha_left, Ha_region_excl, Ha_right]

        #For Hb and [OIII]
        Hb_left = SpectralRegion(lamb[0], Hb_ini - 500 *u.AA)
        Hb_region = SpectralRegion(Hb_ini, Hb_end)
        OIII_region = SpectralRegion(O_ini, O_end)
        OIII_right = SpectralRegion(O_end + 500 *u.AA, lamb[-1])
        Hb_exclusion_regions = [Hb_left, Hb_region, OIII_region, OIII_right]

        #//////// Compute continuum by fitting /////////

        # Halpha
        try:
            Ha_continuum_fit = fit_generic_continuum(clean_spectrum,exclude_regions=Ha_exclusion_regions)(clean_spectrum.spectral_axis)
        except Exception:
            Ha_continuum_fit = None

        # Hbeta
        try:
            Hb_continuum_fit = fit_generic_continuum(clean_spectrum,exclude_regions=Hb_exclusion_regions)(clean_spectrum.spectral_axis)
        except Exception:
            Hb_continuum_fit = None

        #////// Compute Equivalent Width ///////////
        if Ha_continuum_fit is not None:    
            Ha_normalized_continuum_spec = clean_spectrum / Ha_continuum_fit
            Ha_ew= equivalent_width(Ha_normalized_continuum_spec, continuum=1, regions=Ha_region)
            Ha_EW = Ha_ew.value
            Ha_EW_err = Ha_ew.uncertainty  
        else:
            Ha_EW = np.nan
            Ha_EW_err = np.nan
            
        
        if Hb_continuum_fit is not None:
            Hb_normalized_continuum_spec = clean_spectrum / Hb_continuum_fit
            Hb_ew = equivalent_width(Hb_normalized_continuum_spec, continuum=1, regions=Hb_region)
            O3_ew = equivalent_width(Hb_normalized_continuum_spec, continuum=1, regions=OIII_region)
            Hb_EW = Hb_ew.value
            Hb_EW_err = Hb_ew.uncertainty
        else:
            Ha_EW = np.nan
            Ha_EW_err = np.nan
        
    
    

print(Hb_EW, Hb_EW_err)


-512.4116110230852 nan Angstrom


/home/nicolas/Programs/miniconda3/envs/spec_env/lib/python3.11/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nicolas/Programs/miniconda3/envs/spec_env/lib/python3.11/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/nicolas/Programs/miniconda3/envs/spec_env/lib/python3.11/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [11]:
index = 0

jades_file = jades_folder / JADES_files[index]
mast_file = mast_folder / MAST_files[index]
z = z_array[index]


with fits.open(jades_file) as j_f:
    #Extract data from the file
    specdata=j_f[1].data

    #Compute rest wavelength
    lambda_obs = specdata['WAVELENGTH'] * 1e-6 / 1e-10 #convert from um to AA.
    lambda_rest = lambda_obs / (1.+float(z))
    lambda_rest_Angstrom = lambda_rest * u.AA #u.AA just includes the units

    #Extract and convert flux from working units to Jy
    flux = specdata['FLUX'] #flux in erg cm-2 s-1 AA-1'
    flux_Jy = flux * lambda_obs**2 / 2.99792458e-5 *u.Jy

    #Uncertainty of the flux
    flux_err = specdata['FLUX_ERR'] * lambda_obs**2 / 2.99792458e-5
    flux_err_Jy = flux_err * u.Jy
    flux_uncertainty = StdDevUncertainty(flux_err_Jy)
    
    #Uncertainty of the flux from std of vector of flux
    flux_err_2_erg = std_flux(lambda_rest, flux, Ha_ini.value - 500, Ha_ini.value, Ha_end.value, Ha_end.value + 500 )
    flux_err_2 = flux_err_2_erg * lambda_obs**2 / 2.99792458e-5
    flux_err_Jy2 = flux_err_2 * np.ones(len(flux)) * u.Jy
    flux_uncertainty_2 = StdDevUncertainty(flux_err_Jy2)


    #Define the spectrum over which the EW will be calculated, use class Spectrum1D from specutils
    spectrum = Spectrum(spectral_axis=lambda_rest_Angstrom,
                                flux=flux_Jy,
                                uncertainty=flux_uncertainty_2)


    #Remove NaN values
    mask = np.isfinite(spectrum.flux.value)
    clean_spectrum = Spectrum(spectral_axis=lambda_rest_Angstrom[mask],
                            flux=flux_Jy[mask],
                            uncertainty=flux_uncertainty[mask])


    with warnings.catch_warnings(): #ignore warnings
        warnings.simplefilter('ignore')

        #///////// Regions to exlude /////////

        lamb = lambda_rest_Angstrom
        #For Ha
        Ha_left = SpectralRegion(lamb[0], Ha_ini - 500 *u.AA)
        Ha_region = SpectralRegion(Ha_ini, Ha_end)
        Ha_region_excl = SpectralRegion(Ha_ini, Ha_end + 150 *u.AA )
        Ha_right = SpectralRegion(Ha_end + 350 *u.AA, lamb[-1])
        Ha_exclusion_regions = [Ha_left, Ha_region_excl, Ha_right]

        #For Hb and [OIII]
        Hb_left = SpectralRegion(lamb[0], Hb_ini - 500 *u.AA)
        Hb_region = SpectralRegion(Hb_ini, Hb_end)
        OIII_region = SpectralRegion(O_ini, O_end)
        OIII_right = SpectralRegion(O_end + 500 *u.AA, lamb[-1])
        Hb_exclusion_regions = [Hb_left, Hb_region, OIII_region, OIII_right]

        #//////// Compute continuum by fitting /////////

        Ha_continuum_fit = fit_generic_continuum(clean_spectrum, exclude_regions=Ha_exclusion_regions)(clean_spectrum.spectral_axis)
        Hb_continuum_fit = fit_generic_continuum(clean_spectrum, exclude_regions=Hb_exclusion_regions)(clean_spectrum.spectral_axis)


        #////// Normalize the spectrum by its continuum ///////////

        Ha_normalized_continuum_spec = clean_spectrum / Ha_continuum_fit
        Hb_normalized_continuum_spec = clean_spectrum / Hb_continuum_fit

        #////// Compute Equivalent Width ///////////

        Ha_EW = equivalent_width(Ha_normalized_continuum_spec, continuum=1, regions=Ha_region)
        Hb_EW = equivalent_width(Hb_normalized_continuum_spec, continuum=1, regions=Hb_region)
        OIII_EW = equivalent_width(Hb_normalized_continuum_spec, continuum=1, regions=OIII_region)

print(ID_array[index])
print(Ha_EW.value, Ha_EW.uncertainty)


3184
-277.7865609371252 8.905188586836523 Angstrom
